# 06 — Finite sampling

Samples the optimized ideal-QAOA distributions at fixed parameters to quantify measurement uncertainty independently of parameter optimization.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from oec_qaoa.benchmarks import (
    FROZEN_QAOA_BENCHMARKS,
    build_frozen_benchmark,
    physical_feasibility_mask,
)
from oec_qaoa.finite_sampling import (
    exact_sampling_baseline,
    optimum_detection_probability,
    sample_distribution,
)
from oec_qaoa.qaoa import (
    basis_energies,
    optimize_qaoa,
    qaoa_statevector,
)

DEPTHS = [1, 2, 3, 4]
SHOTS = [128, 512, 2048, 8192]
REPETITIONS = 30
MAXITER = 200

BEST_SEEDS = {
    ("B1", 1): 101,
    ("B1", 2): 404,
    ("B1", 3): 303,
    ("B1", 4): 404,
    ("B2", 1): 303,
    ("B2", 2): 303,
    ("B2", 3): 404,
    ("B2", 4): 404,
    ("B3", 1): 202,
    ("B3", 2): 404,
    ("B3", 3): 303,
    ("B3", 4): 404,
}

## Sampling protocol

Shot budgets are `128`, `512`, `2048` and `8192`, with 30 deterministic repetitions per configuration. For ideal optimal probability `P_opt`, the probability of observing at least one optimum is

\[
P_{\mathrm{detect}}=1-(1-P_{\mathrm{opt}})^{N_{\mathrm{shots}}}.
\]

In [ ]:
def cost_scale(model) -> float:
    coefficients = [
        abs(float(value))
        for value in model.linear
        if abs(float(value)) > 1e-12
    ]
    coefficients.extend(
        abs(float(value))
        for value in model.quadratic.values()
        if abs(float(value)) > 1e-12
    )
    return max(coefficients) if coefficients else 1.0


records = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, retained, model = build_frozen_benchmark(benchmark)
    feasible_mask = physical_feasibility_mask(instance, retained, model)

    energies = basis_energies(model)
    optimum = float(energies.min())
    optimal_mask = np.isclose(energies, optimum, atol=1e-9)
    phase_energies = energies / cost_scale(model)

    for depth in DEPTHS:
        seed = BEST_SEEDS[(benchmark.benchmark_id, depth)]
        result = optimize_qaoa(
            model,
            depth=depth,
            seed=seed,
            feasible_mask=feasible_mask,
            optimizer="COBYLA",
            maxiter=MAXITER,
            normalize_cost=True,
        )

        state = qaoa_statevector(
            phase_energies,
            result.parameters[:depth],
            result.parameters[depth:],
        )
        probabilities = np.abs(state) ** 2
        baseline = exact_sampling_baseline(
            probabilities,
            energies,
            optimal_mask,
            feasible_mask,
        )

        for shots in SHOTS:
            for repetition in range(REPETITIONS):
                sample_seed = (
                    1_000_000
                    + 10_000 * int(benchmark.benchmark_id[1:])
                    + 1_000 * depth
                    + 10 * SHOTS.index(shots)
                    + repetition
                )
                sampled = sample_distribution(
                    probabilities,
                    energies,
                    optimal_mask,
                    feasible_mask,
                    shots=shots,
                    seed=sample_seed,
                )

                records.append(
                    {
                        "benchmark_id": benchmark.benchmark_id,
                        "depth": depth,
                        "shots": shots,
                        "repetition": repetition,
                        "sampled_expected_energy": (
                            sampled.sampled_expected_energy
                        ),
                        "sampled_optimal_probability": (
                            sampled.sampled_optimal_probability
                        ),
                        "sampled_feasible_probability": (
                            sampled.sampled_feasible_probability
                        ),
                        "best_sampled_energy": sampled.best_sampled_energy,
                        "optimum_observed": sampled.optimum_observed,
                        "ideal_expected_energy": baseline[
                            "exact_expected_energy"
                        ],
                        "ideal_optimal_probability": baseline[
                            "exact_optimal_probability"
                        ],
                        "ideal_feasible_probability": baseline[
                            "exact_feasible_probability"
                        ],
                        "analytic_optimum_detection_probability": (
                            optimum_detection_probability(
                                baseline["exact_optimal_probability"],
                                shots,
                            )
                        ),
                    }
                )

runs = pd.DataFrame(records)
runs.head()

## Sampling statistics

For each shot budget we record sampled energy, optimal and feasible probabilities, the best sampled energy and whether an optimum was observed.

In [ ]:
summary = (
    runs.groupby(["benchmark_id", "depth", "shots"])
    .agg(
        sampled_expected_energy_mean=(
            "sampled_expected_energy",
            "mean",
        ),
        sampled_expected_energy_std=(
            "sampled_expected_energy",
            "std",
        ),
        sampled_optimal_probability_mean=(
            "sampled_optimal_probability",
            "mean",
        ),
        sampled_feasible_probability_mean=(
            "sampled_feasible_probability",
            "mean",
        ),
        optimum_observation_rate=("optimum_observed", "mean"),
        best_sampled_energy_mean=("best_sampled_energy", "mean"),
        ideal_expected_energy=("ideal_expected_energy", "first"),
        ideal_optimal_probability=(
            "ideal_optimal_probability",
            "first",
        ),
        ideal_feasible_probability=(
            "ideal_feasible_probability",
            "first",
        ),
        analytic_optimum_detection_probability=(
            "analytic_optimum_detection_probability",
            "first",
        ),
    )
    .reset_index()
)

summary

## Result summary

Energy estimates converge faster than exact-optimum detection when `P_opt` is small. Feasible allocations are sampled more frequently than globally optimal ones in the harder benchmarks.